### Apply Churn Model to Generate Risk Scores

For existing members (2017 history available): use the trained model.
For new 2018 enrollees: no 2017 history exists, so we mark them as
"Not Scored - New Member" rather than producing a misleading prediction.

In [1]:
import pandas as pd
import numpy as np
import joblib

# Load model and expected columns
rf = joblib.load("../src/model/churn_rf_model.pkl")
model_columns = joblib.load("../src/model/model_columns.pkl")

# Load datasets
segments = pd.read_csv("../data/processed/customer_segments_final.csv")
print(segments.shape)

(16737, 33)


In [2]:
# Separate existing members (have real 2017 data) vs new 2018 enrollees
existing = segments[segments['Enrollment Year'] != 2018].copy()
new_members = segments[segments['Enrollment Year'] == 2018].copy()

print("Existing:", existing.shape[0], "New:", new_members.shape[0])

Existing: 13727 New: 3010


In [3]:
# Recreate the exact same feature prep as in notebook 03
drop_cols = ['Loyalty Number', 'Country', 'churned']
X_existing = existing.drop(columns=[c for c in drop_cols if c in existing.columns])

cat_cols = X_existing.select_dtypes(include='object').columns.tolist()
X_encoded = pd.get_dummies(X_existing, columns=cat_cols, drop_first=True)

# Align columns to match training (add missing cols as 0, drop extras, reorder)
X_encoded = X_encoded.reindex(columns=model_columns, fill_value=0)

print(X_encoded.shape)

(13727, 39)


C:\Users\hp\AppData\Local\Temp\ipykernel_1012\914306808.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_existing.select_dtypes(include='object').columns.tolist()


In [4]:
# Predict churn probability
risk_scores = rf.predict_proba(X_encoded)[:, 1]
existing['churn_risk'] = risk_scores

print(existing['churn_risk'].describe())

count    13727.000000
mean         0.306773
std          0.269738
min          0.043351
25%          0.140777
50%          0.173601
75%          0.384562
max          0.998931
Name: churn_risk, dtype: float64


In [5]:
# New members: not scorable (no behavioral history) -> mark as -1 (we'll handle in dashboard)
new_members['churn_risk'] = -1

# Combine back together
final = pd.concat([existing, new_members], ignore_index=True)

final.to_csv("../data/processed/customer_segments_final.csv", index=False)
print("Saved with churn_risk column:", final.shape)
print(final[['Loyalty Number', 'final_segment', 'churn_risk']].head())

Saved with churn_risk column: (16737, 34)
   Loyalty Number final_segment  churn_risk
0          480934     segment_3    0.156979
1          549612     segment_0    0.168689
2          429460     segment_3    0.190930
3          608370     segment_0    0.131538
4          530508     segment_3    0.164430
